# Classificação de Categorias de Notícias

Neste notebook é apresentado o processo de desenvolvimento do modelo de classificação de notícias, incluindo carregamento dos dados pré-processados, treinamento, avaliação e persistência do modelo final.

In [96]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from datetime import datetime

In [99]:
df_filtered = pd.read_csv('../data/filtered_articles.csv')

In [100]:
print(df_filtered.shape)
df_filtered.head()

(167044, 3)


,title,text,category
0,"Lula diz que está 'lascado', mas que ainda tem...",Com a possibilidade de uma condenação impedir ...,poder
1,"'Decidi ser escrava das mulheres que sofrem', ...","Para Oumou Sangaré, cantora e ativista malines...",ilustrada
2,Três reportagens da Folha ganham Prêmio Petrob...,Três reportagens da Folha foram vencedoras do ...,poder
3,Filme 'Star Wars: Os Últimos Jedi' ganha trail...,A Disney divulgou na noite desta segunda-feira...,ilustrada
4,CBSS inicia acordos com fintechs e quer 30% do...,"O CBSS, banco da holding Elopar dos sócios Bra...",mercado


## Construção da entrada do modelo

Para cada notícia, foi criado um único documento textual concatenando o título e o corpo da notícia, conforme o exemplo abaixo:

```
entrada = title + ". " + text
```

Essa estratégia permite que o modelo utilize tanto a informação resumida presente no título quanto o contexto fornecido pelo corpo da notícia, aumentando a quantidade de informação disponível para a classificação.

In [101]:
df_filtered["input"] = (
    df_filtered["title"].fillna("") + ". " +
    df_filtered["text"].fillna("")
).str.strip()

In [94]:
X = df_filtered["input"]
y = df_filtered['category']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=df_filtered["category"],
    random_state=42
)

In [102]:
print(f'Tamanho dos dados de treino: {len(X_train)}')
print(f'Tamanho dos dados de teste: {len(X_test)}')

Tamanho dos dados de treino: 133635
Tamanho dos dados de teste: 33409


### Vetorização

Os textos foram convertidos para representação numérica utilizando TF-IDF.

Essa técnica atribui maior importância a termos relevantes para cada documento e costuma ser utilizada em tarefas clássicas de classificação de textos.

### Modelo

Foi utilizado um LinearSVC.

Esse algoritmo costuma apresentar bom desempenho em problemas de classificação de textos utilizando representações esparsas como TF-IDF, além de possuir baixo custo computacional.

### Escoha dos hiperparâmetros

Os hiperparâmetros apresentados representam uma configuração que equilibra desempenho computacional e qualidade das predições.

Durante os experimentos, foram avaliadas configurações mais complexas, como o uso de bigramas (ngram_range=(1,2)) e um vocabulário maior (max_features=50000). Entretanto, essas configurações aumentaram significativamente o consumo de memória e o tempo de processamento, tornando o treinamento inviável no ambiente computacional disponível.

Dessa forma, optou-se por utilizar apenas unigramas e limitar o vocabulário a 20.000 termos, permitindo concluir o treinamento em tempo hábil e obter um modelo com bom desempenho.

In [52]:
pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1,1),
            min_df=3,
            max_df=0.95,
            max_features=20000
        )
    ),
    (
        "classifier",
        LinearSVC( 
            random_state=42,
            max_iter=200
        )
    )
])

pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, max_features=20000, min_df=3)),
                ('classifier', LinearSVC(max_iter=200, random_state=42))])

In [53]:
# Testando o modelo
y_pred = pipeline.predict(X_test)

## Avaliação

O modelo foi avaliado utilizando um conjunto de teste separado durante o treinamento.

As métricas analisadas foram:

- Acurácia;
- Precisão;
- Revocação;
- F1-score.

In [59]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro', zero_division=0) 
recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

print(f"Acurácia:  {accuracy:.2f}")   
print(f"Precisão:  {precision:.2f}") 
print(f"Revocação: {recall:.2f}")    
print(f"F1-score:  {f1:.2f}")  

Acurácia:  0.87
Precisão:  0.58
Revocação: 0.46
F1-score:  0.49


In [54]:
print(classification_report(y_test, y_pred, zero_division=0) )

                              precision    recall  f1-score   support

                    ambiente       0.49      0.47      0.48        98
                      asmais       0.45      0.08      0.14       110
              banco-de-dados       1.00      0.08      0.14        13
                         bbc       0.67      0.31      0.42       196
               cenarios-2017       0.00      0.00      0.00         9
                     ciencia       0.68      0.64      0.66       267
                     colunas       0.84      0.84      0.84      4324
                      comida       0.68      0.63      0.65       166
                   cotidiano       0.86      0.89      0.87      3393
                          dw       0.00      0.00      0.00        10
                    educacao       0.78      0.87      0.82       424
          empreendedorsocial       0.81      0.73      0.76       168
            equilibrioesaude       0.65      0.66      0.65       262
                   

## Discussão dos resultados

O modelo apresentou bom desempenho geral, atingindo uma acurácia de aproximadamente 87%.

Observa-se, entretanto, uma diferença significativa entre o F1-score Macro e o F1-score Weighted. Esse comportamento é esperado devido ao forte desbalanceamento entre as categorias do conjunto de dados, fazendo com que classes pouco representadas apresentem desempenho inferior.

Por outro lado, categorias com maior número de amostras apresentaram métricas superiores a 0,85, indicando que o modelo consegue capturar adequadamente os padrões presentes nessas classes.

In [66]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"../models/news_classifier_f1_{f1:.2f}_{timestamp}.pkl"
joblib.dump(pipeline, filename)

print(f"Modelo salvo em: {filename}")

Modelo salvo em: ../models/news_classifier_f1_0.49_20260711_104852.pkl
